# 03 — Momentum Baseline Evaluation

Evaluate 12–2 momentum **before** conditioning on market concentration.

Primary baseline: monthly Rank IC and value-weighted quintile portfolios. Equal-weighted returns are shown as a robustness comparison. Formal HAC t-statistics are deferred until the Newey–West lag rule is frozen.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel, baseline_summary


## 1. Rebuild the validated momentum dataset


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
mom = attach_momentum_to_formation_universe(clean, history)
mom = assign_signal_quintiles(mom)
print(f'Rows: {len(mom):,}')
print(f'Valid momentum: {mom.Momentum_12_2.notna().sum():,}')
print(f'Valid next-month returns: {mom.NextMthRet.notna().sum():,}')


## 2. Monthly Rank IC


In [ ]:
rank_ic = monthly_rank_ic(mom)
rank_ic.tail(12)


In [ ]:
rank_ic['RankIC'].describe()


## 3. Quintile portfolio returns and missing-return coverage


In [ ]:
qret = quintile_returns(mom)
qret.tail(10)


In [ ]:
coverage_check = qret.groupby('SignalQuintile', observed=True)[
    ['ObservedReturn_CountShare', 'ObservedReturn_CapShare']
].agg(['mean','min'])
coverage_check


## 4. Value-weighted baseline (primary)


In [ ]:
vw = quintile_return_panel(qret, return_col='VW_Return')
vw.tail()


In [ ]:
vw_means = vw[['Q1','Q2','Q3','Q4','Q5','Q5_minus_Q1']].mean()
vw_means


## 5. Equal-weighted baseline (robustness)


In [ ]:
ew = quintile_return_panel(qret, return_col='EW_Return')
ew_means = ew[['Q1','Q2','Q3','Q4','Q5','Q5_minus_Q1']].mean()
ew_means


## 6. Descriptive baseline summary


In [ ]:
summary = baseline_summary(rank_ic, vw)
summary


## 7. Visual sanity checks


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(rank_ic['MthCalDt'], rank_ic['RankIC'])
ax.axhline(0, linewidth=1)
ax.set_title('Monthly momentum Rank IC')
ax.set_xlabel('Month')
ax.set_ylabel('Spearman Rank IC')
plt.show()


In [ ]:
plot_data = vw.dropna(subset=['Q5_minus_Q1']).copy()
plot_data['GrossGrowth'] = (1 + plot_data['Q5_minus_Q1']).cumprod()
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(plot_data['MthCalDt'], plot_data['GrossGrowth'])
ax.set_title('Gross growth of $1: value-weighted Q5 − Q1 momentum')
ax.set_xlabel('Month')
ax.set_ylabel('Gross growth')
plt.show()


## Stop before concentration conditioning

Interpret the baseline and coverage diagnostics first. Do not condition on concentration until the baseline signal behaviour is understood and the inference convention is fixed.
